# Brain Tumor Detection

Binary classification of MRI brain scans (tumor / no tumor) using four deep learning approaches:

1. **VGG-16** — transfer learning with brain crop preprocessing (Keras)
2. **VGG-19 & ResNet-50** — pretrained ImageNet models (Keras)
3. **EfficientNetB3** — lightweight efficient architecture (TensorFlow)
4. **Custom CNN** — built from scratch in PyTorch

**Dataset:** Brain MRI Images for Brain Tumor Detection  
**Classes:** `yes` (tumor present) · `no` (no tumor)  
**Dataset path:** `/content/drive/MyDrive/BTD_Notebook/brain_tumor_dataset`


## 1. Setup & Installs

In [ ]:
!pip install -q imutils tensorflow==2.9.1 torch torchvision
from IPython.display import clear_output
clear_output()

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import shutil
import itertools
import imutils
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelBinarizer
import tensorflow as tf
import keras
from keras.models import Sequential
from keras import layers
from keras.applications import VGG16, VGG19, ResNet50
from keras.applications.vgg16 import preprocess_input
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import EarlyStopping
from keras.optimizers import RMSprop, Adamax
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/content/drive/MyDrive/BTD_Notebook/brain_tumor_dataset'
IMG_SIZE = (224, 224)

## 2. Load & Explore Data

In [ ]:
# Count images per class
for cls in ['yes', 'no']:
    count = len(os.listdir(os.path.join(DATA_DIR, cls)))
    print(f"{cls}: {count} images")

# Show sample MRI scans
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, cls in enumerate(['yes', 'no']):
    cls_path = os.path.join(DATA_DIR, cls)
    files = os.listdir(cls_path)[:5]
    for j, fname in enumerate(files):
        img = cv2.imread(os.path.join(cls_path, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i][j].imshow(img)
        axes[i][j].axis('off')
        axes[i][j].set_title(f'{"Tumor" if cls == "yes" else "No Tumor"}', fontsize=10)

plt.suptitle('Sample MRI Brain Scans', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Preprocessing

Brain MRI images are cropped to remove empty space outside the brain using contour detection.
This improves model focus and reduces noise.


In [ ]:
def load_data(dir_path, img_size=(100, 100)):
    """Load resized images as np.arrays."""
    X, y = [], []
    labels = {}
    i = 0
    for path in tqdm(sorted(os.listdir(dir_path))):
        if not path.startswith('.'):
            labels[i] = path
            for file in os.listdir(os.path.join(dir_path, path)):
                if not file.startswith('.'):
                    img = cv2.imread(os.path.join(dir_path, path, file))
                    if img is not None:
                        img = cv2.resize(img, img_size)
                        X.append(img)
                        y.append(i)
            i += 1
    return np.array(X), np.array(y), labels


def crop_brain(img):
    """Crop brain MRI to remove empty border space using contour detection."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)
    cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)
    if not cnts:
        return img
    c = max(cnts, key=cv2.contourArea)
    extLeft   = tuple(c[c[:, :, 0].argmin()][0])
    extRight  = tuple(c[c[:, :, 0].argmax()][0])
    extTop    = tuple(c[c[:, :, 1].argmin()][0])
    extBottom = tuple(c[c[:, :, 1].argmax()][0])
    return img[extTop[1]:extBottom[1], extLeft[0]:extRight[0]]


def crop_imgs(set_name):
    return [crop_brain(img) for img in set_name]


def preprocess_imgs(set_name, img_size):
    """Resize and apply VGG preprocessing."""
    return np.array([
        preprocess_input(cv2.resize(img, dsize=img_size, interpolation=cv2.INTER_CUBIC))
        for img in set_name
    ])

In [ ]:
# Visualize the crop pipeline on one image
sample_path = os.path.join(DATA_DIR, 'yes/Y105.jpg')
img = cv2.imread(sample_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_CUBIC)

gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
gray = cv2.GaussianBlur(gray, (5, 5), 0)
thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
thresh = cv2.erode(thresh, None, iterations=2)
thresh = cv2.dilate(thresh, None, iterations=2)
cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts = imutils.grab_contours(cnts)
c = max(cnts, key=cv2.contourArea)
cropped = crop_brain(img_resized)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_resized); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(thresh, cmap='gray'); axes[1].set_title('Threshold Mask'); axes[1].axis('off')
axes[2].imshow(cropped); axes[2].set_title('Cropped Brain'); axes[2].axis('off')
plt.suptitle('Brain Crop Preprocessing Pipeline', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Build TRAIN / VAL / TEST directories from source
!mkdir -p TRAIN/YES TRAIN/NO TEST/YES TEST/NO VAL/YES VAL/NO

for cls in ['yes', 'no']:
    cls_upper = cls.upper()
    files = [f for f in os.listdir(os.path.join(DATA_DIR, cls)) if not f.startswith('.')]
    train_f, test_f = train_test_split(files, test_size=0.2, random_state=42)
    val_f,   test_f = train_test_split(test_f, test_size=0.5, random_state=42)
    for f in train_f: shutil.copy(os.path.join(DATA_DIR, cls, f), f'TRAIN/{cls_upper}/{f}')
    for f in val_f:   shutil.copy(os.path.join(DATA_DIR, cls, f), f'VAL/{cls_upper}/{f}')
    for f in test_f:  shutil.copy(os.path.join(DATA_DIR, cls, f), f'TEST/{cls_upper}/{f}')

# Load arrays
X_train, y_train, labels = load_data('TRAIN/', IMG_SIZE)
X_val,   y_val,   _      = load_data('VAL/',   IMG_SIZE)
X_test,  y_test,  _      = load_data('TEST/',  IMG_SIZE)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# Apply brain crop + VGG preprocessing
X_train_crop = crop_imgs(X_train)
X_val_crop   = crop_imgs(X_val)
X_test_crop  = crop_imgs(X_test)

X_train_prep = preprocess_imgs(X_train_crop, IMG_SIZE)
X_val_prep   = preprocess_imgs(X_val_crop,   IMG_SIZE)
X_test_prep  = preprocess_imgs(X_test_crop,  IMG_SIZE)

# Save cropped sets to disk for ImageDataGenerator
!mkdir -p TRAIN_CROP/YES TRAIN_CROP/NO VAL_CROP/YES VAL_CROP/NO TEST_CROP/YES TEST_CROP/NO

def save_new_images(x_set, y_set, folder_name):
    for i, (img, label) in enumerate(zip(x_set, y_set)):
        cls = 'YES' if label == 1 else 'NO'
        cv2.imwrite(f'{folder_name}{cls}/{i}.jpg', img)

save_new_images(X_train_crop, y_train, 'TRAIN_CROP/')
save_new_images(X_val_crop,   y_val,   'VAL_CROP/')
save_new_images(X_test_crop,  y_test,  'TEST_CROP/')
print("Saved cropped images.")

## 4. Shared Utilities

In [ ]:
def plot_confusion_matrix(cm, classes, normalize=False,
                          title='Confusion Matrix', cmap=plt.cm.Blues):
    plt.figure(figsize=(6, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, [c[1] for c in classes], rotation=45)
    plt.yticks(tick_marks, [c[1] for c in classes])
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment='center',
                 color='white' if cm[i, j] > thresh else 'black')
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()


def plot_training_curves(history, title):
    acc     = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss    = history.history['loss']
    val_loss= history.history['val_loss']
    epochs  = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, acc,     label='Train')
    axes[0].plot(epochs, val_acc, label='Val')
    axes[0].set_title(f'{title} — Accuracy')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(epochs, loss,     label='Train')
    axes[1].plot(epochs, val_loss, label='Val')
    axes[1].set_title(f'{title} — Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 5. Model 1 — VGG-16

Pretrained VGG-16 (ImageNet weights, top excluded) with a custom sigmoid output.  
Uses brain-cropped and preprocessed MRI images. Augmentation applied during training.


In [ ]:
# Data augmentation
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    brightness_range=[0.5, 1.5],
    horizontal_flip=True,
    vertical_flip=True,
    rescale=1./255,
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen_vgg16 = train_datagen.flow_from_directory(
    'TRAIN_CROP', target_size=IMG_SIZE, batch_size=16,
    class_mode='binary', shuffle=True)
val_gen_vgg16 = val_datagen.flow_from_directory(
    'VAL_CROP', target_size=IMG_SIZE, batch_size=16,
    class_mode='binary', shuffle=False)
test_gen_vgg16 = val_datagen.flow_from_directory(
    'TEST_CROP', target_size=IMG_SIZE, batch_size=16,
    class_mode='binary', shuffle=False)

In [ ]:
# Build VGG-16 model
vgg16_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=IMG_SIZE + (3,)
)
vgg16_base.trainable = False

model_vgg16 = Sequential([
    vgg16_base,
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
])
model_vgg16.compile(
    loss='binary_crossentropy',
    optimizer=RMSprop(learning_rate=1e-4),
    metrics=['accuracy']
)
model_vgg16.summary()

In [ ]:
es_vgg16 = EarlyStopping(monitor='val_accuracy', mode='max', patience=6)

history_vgg16 = model_vgg16.fit(
    train_gen_vgg16,
    validation_data=val_gen_vgg16,
    epochs=30,
    callbacks=[es_vgg16],
    verbose=1
)
model_vgg16.save('REU23_VGG16_model.h5')

In [ ]:
plot_training_curves(history_vgg16, 'VGG-16')

# Evaluate
predictions_vgg16 = (model_vgg16.predict(X_test_prep) > 0.5).astype(int)
acc_vgg16 = accuracy_score(y_test, predictions_vgg16)
print(f'VGG-16 Test Accuracy: {acc_vgg16:.4f}')

cm_vgg16 = confusion_matrix(y_test, predictions_vgg16)
plot_confusion_matrix(cm_vgg16, list(labels.items()), title='VGG-16 Confusion Matrix')

## 6. Model 2 — VGG-19

VGG-19 is deeper than VGG-16 and can capture more intricate patterns,
at the cost of greater computational resources and higher risk of overfitting on small datasets.


In [ ]:
image_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_set_vgg19 = image_gen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=16,
    class_mode='binary', subset='training')
test_set_vgg19 = image_gen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=16,
    class_mode='binary', subset='validation')

In [ ]:
vgg19_base = VGG19(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
for layer in vgg19_base.layers:
    layer.trainable = False

x = layers.Flatten()(vgg19_base.output)
x = layers.Dropout(0.4)(x)
x = layers.Dense(1, activation='sigmoid')(x)

model_vgg19 = keras.Model(vgg19_base.input, x)
model_vgg19.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_vgg19.summary()

In [ ]:
history_vgg19 = model_vgg19.fit(
    train_set_vgg19,
    validation_data=test_set_vgg19,
    epochs=30,
    verbose=1
)
model_vgg19.save('VGG19_model.h5')

In [ ]:
plot_training_curves(history_vgg19, 'VGG-19')

train_acc_vgg19 = history_vgg19.history['accuracy'][-1]
val_acc_vgg19   = history_vgg19.history['val_accuracy'][-1]
print(f'VGG-19 Final Training Accuracy:   {train_acc_vgg19:.4f}')
print(f'VGG-19 Final Validation Accuracy: {val_acc_vgg19:.4f}')

y_pred_vgg19 = (model_vgg19.predict(test_set_vgg19) > 0.5).astype(int).flatten()
cm_vgg19 = confusion_matrix(test_set_vgg19.classes, y_pred_vgg19)
plot_confusion_matrix(cm_vgg19, [(0,'no'), (1,'yes')], title='VGG-19 Confusion Matrix')

## 7. Model 3 — ResNet-50

In [ ]:
resnet_base = ResNet50(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
for layer in resnet_base.layers:
    layer.trainable = False

x = layers.Flatten()(resnet_base.output)
x = layers.Dropout(0.4)(x)
x = layers.Dense(1, activation='sigmoid')(x)

model_resnet = keras.Model(resnet_base.input, x)
model_resnet.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_resnet.summary()

In [ ]:
history_resnet = model_resnet.fit(
    train_set_vgg19,        # reuse same generators
    validation_data=test_set_vgg19,
    epochs=30,
    verbose=1
)
model_resnet.save('RESNET50_model.h5')

In [ ]:
plot_training_curves(history_resnet, 'ResNet-50')

train_acc_resnet = history_resnet.history['accuracy'][-1]
val_acc_resnet   = history_resnet.history['val_accuracy'][-1]
print(f'ResNet-50 Final Training Accuracy:   {train_acc_resnet:.4f}')
print(f'ResNet-50 Final Validation Accuracy: {val_acc_resnet:.4f}')

y_pred_resnet = (model_resnet.predict(test_set_vgg19) > 0.5).astype(int).flatten()
cm_resnet = confusion_matrix(test_set_vgg19.classes, y_pred_resnet)
plot_confusion_matrix(cm_resnet, [(0,'no'), (1,'yes')], title='ResNet-50 Confusion Matrix')

## 8. Model 4 — EfficientNetB3

EfficientNet scales depth, width, and resolution together for better accuracy
with fewer parameters. Uses `categorical` class mode for multi-class compatibility.


In [ ]:
# Build dataframe of file paths + labels
filepaths, file_labels = [], []
for cls in os.listdir(DATA_DIR):
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        for file in os.listdir(cls_path):
            if not file.startswith('.'):
                filepaths.append(os.path.join(cls_path, file))
                file_labels.append(cls)

df = pd.DataFrame({'filepaths': filepaths, 'labels': file_labels})
train_df, test_df  = train_test_split(df, train_size=0.8, shuffle=True, random_state=123)
valid_df, test_df  = train_test_split(test_df, train_size=0.5, shuffle=True, random_state=123)
print(f"Train: {len(train_df)} | Val: {len(valid_df)} | Test: {len(test_df)}")

In [ ]:
batch_size = 16
tr_gen = ImageDataGenerator()
ts_gen = ImageDataGenerator()

train_gen_eff = tr_gen.flow_from_dataframe(
    train_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    color_mode='rgb', shuffle=True, batch_size=batch_size)
valid_gen_eff = ts_gen.flow_from_dataframe(
    valid_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    color_mode='rgb', shuffle=False, batch_size=batch_size)
test_gen_eff = ts_gen.flow_from_dataframe(
    test_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    color_mode='rgb', shuffle=False, batch_size=batch_size)

In [ ]:
class_count = len(train_gen_eff.class_indices)

eff_base = tf.keras.applications.EfficientNetB3(
    include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,), pooling='max')

model_eff = Sequential([
    eff_base,
    layers.BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001),
    layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.016),
                 activity_regularizer=tf.keras.regularizers.l1(0.006),
                 bias_regularizer=tf.keras.regularizers.l1(0.006), activation='relu'),
    layers.Dropout(rate=0.45, seed=123),
    layers.Dense(class_count, activation='softmax'),
])
model_eff.compile(
    optimizer=Adamax(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_eff.summary()

In [ ]:
history_eff = model_eff.fit(
    train_gen_eff,
    epochs=50,
    validation_data=valid_gen_eff,
    shuffle=False,
    verbose=1
)
model_eff.save('BTDEffNET.h5')

In [ ]:
plot_training_curves(history_eff, 'EfficientNetB3')

# Evaluate
train_score = model_eff.evaluate(train_gen_eff, verbose=0)
valid_score = model_eff.evaluate(valid_gen_eff, verbose=0)
test_score  = model_eff.evaluate(test_gen_eff,  verbose=0)
print(f'Train Accuracy: {train_score[1]:.4f}')
print(f'Val   Accuracy: {valid_score[1]:.4f}')
print(f'Test  Accuracy: {test_score[1]:.4f}')

preds_eff = model_eff.predict(test_gen_eff)
y_pred_eff = np.argmax(preds_eff, axis=1)
classes_eff = list(test_gen_eff.class_indices.keys())

cm_eff = confusion_matrix(test_gen_eff.classes, y_pred_eff)
plot_confusion_matrix(cm_eff, list(enumerate(classes_eff)), title='EfficientNetB3 Confusion Matrix')
print(classification_report(test_gen_eff.classes, y_pred_eff, target_names=classes_eff))

## 9. Model 5 — Custom CNN (PyTorch)

A custom 5-block convolutional neural network built from scratch in PyTorch.
Trained on grayscale MRI images using cross-entropy loss and Adam optimizer.


In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(),
    transforms.ToTensor(),
])

dataset    = datasets.ImageFolder(DATA_DIR, train_transforms)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
print('Classes:', dataset.classes)
print('Total samples:', len(dataset))

In [ ]:
def conv_block(ni, no):
    return nn.Sequential(
        nn.Conv2d(ni, no, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.BatchNorm2d(no),
        nn.Conv2d(no, no, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.BatchNorm2d(no),
        nn.MaxPool2d(2),
    )

class BrainTumorCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1   = conv_block(1, 32)
        self.conv2   = conv_block(32, 64)
        self.conv3   = conv_block(64, 128)
        self.conv4   = conv_block(128, 256)
        self.conv5   = conv_block(256, 512)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(512, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)

model_cnn = BrainTumorCNN().to(device)
print(model_cnn)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=0.001)

num_epochs   = 20
train_losses = []
train_accs   = []

for epoch in range(num_epochs):
    total_loss, total_correct, total_samples = 0, 0, 0
    for images, labels_batch in dataloader:
        images, labels_batch = images.to(device), labels_batch.to(device)
        outputs = model_cnn(images)
        loss    = criterion(outputs, labels_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels_batch).sum().item()
        total_samples += images.size(0)
    epoch_loss = total_loss / total_samples
    epoch_acc  = total_correct / total_samples
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    print(f'Epoch [{epoch+1}/{num_epochs}]  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')

torch.save(model_cnn.state_dict(), 'brain_tumor_cnn.pth')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs_range = range(1, num_epochs + 1)
axes[0].plot(epochs_range, train_accs, marker='o', label='Train Accuracy')
axes[0].set_title('Custom CNN — Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy'); axes[0].legend()
axes[1].plot(epochs_range, train_losses, marker='o', color='orange', label='Train Loss')
axes[1].set_title('Custom CNN — Loss'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss'); axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Inference on a Single Image

In [ ]:
from keras.models import load_model

def predict_mri(image_path, model, img_size=(224, 224), use_preprocess=True):
    """
    Run inference on a single MRI image.
    Returns predicted class label and confidence score.
    """
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, img_size, interpolation=cv2.INTER_CUBIC)
    if use_preprocess:
        img = preprocess_input(img)
    img_array = np.expand_dims(img, axis=0)
    pred = model.predict(img_array, verbose=0)[0][0]
    label = 'Tumor Detected' if pred > 0.5 else 'No Tumor'
    return label, pred

# Test on a sample image
test_img = os.path.join(DATA_DIR, 'yes/Y1.jpg')
label, confidence = predict_mri(test_img, model_vgg16)

img_display = cv2.cvtColor(cv2.imread(test_img), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(5, 5))
plt.imshow(img_display)
plt.axis('off')
plt.title(f'{label}\nConfidence: {confidence:.2%}', fontsize=13)
plt.show()

## 11. Results Summary

| Model | Framework | Accuracy Range | Notes |
|---|---|---|---|
| VGG-16 | Keras/TF | 75–90% | Best balance; brain crop preprocessing |
| VGG-19 | Keras/TF | 75–90% | Higher capacity; prone to overfitting on small datasets |
| ResNet-50 | Keras/TF | 75–90% | Skip connections; strong generalization |
| EfficientNetB3 | TensorFlow | 75–90% | Most parameter-efficient |
| Custom CNN (Keras) | Keras/TF | 75–90% | High-level API baseline |

Testing accuracy ranged from **75% to 90%** across all models, presented at the 35th GMIS Conference (CAHSI).  
Performance variation suggests further optimization and larger datasets could yield more consistent results.

**Key observations:**
- Brain crop preprocessing (contour detection → extreme point cropping) consistently improves focus on relevant tissue
- VGG-19 is more prone to overfitting than VGG-16 on this dataset size
- EfficientNetB3 achieves competitive accuracy with fewer parameters than VGG variants
- Diversity in architectures enables comprehensive cross-model comparison

**Future work (from paper):**
- Validate on additional brain tumor datasets
- Extend to other MRI imaging tasks (lung damage, bone fractures)

